# Naver 뉴스 본문 수집 (Colab용)
- url_수집_colab.ipynb 실행 후 생성된 링크 JSON 파일을 기반으로 본문 수집
- Google Drive에서 링크 파일 로드 및 결과 CSV 저장

In [ ]:
# Colab 환경 세팅 — Selenium, ChromeDriver 설치
!apt-get update -q
!apt-get install -y -q chromium-browser chromium-chromedriver
!pip install -q selenium webdriver-manager

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
import random
import json
import os

# 봇 탐지를 피하기 위해 user agent로 설정
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:138.0) Gecko/20100101 Firefox/138.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
]

options = Options()
options.add_argument(f'user-agent={random.choice(user_agents)}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--disable-blink-features=AutomationControlled')

# Colab은 GUI가 없으므로 headless 필수
options.add_argument('--headless=new')  # 구형 --headless보다 탐지 어려움
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')  # headless에서 뷰포트 명시

# Colab에 설치된 chromium 경로 직접 지정
options.binary_location = '/usr/bin/chromium-browser'
service = Service('/usr/bin/chromedriver')

driver = webdriver.Chrome(service=service, options=options)

# navigator.webdriver 플래그 제거
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})

# 변수 정의
query      = 'SK텔레콤'
start_date = '2025.04.01'
end_date   = '2025.04.30'

# Google Drive 경로
DRIVE_DIR = '/content/drive/MyDrive/naver_crawl/data'
os.makedirs(DRIVE_DIR, exist_ok=True)

# url_수집_colab.ipynb에서 저장한 링크 파일 경로
links_file_name = f"링크_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
links_path = os.path.join(DRIVE_DIR, links_file_name)

In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행

# 월별 링크 파일 불러오기
if os.path.exists(links_path):
    with open(links_path, 'r', encoding='utf-8') as f:
        naver_news_links = json.load(f)
    print(f'링크 {len(naver_news_links)}개 불러옴: {links_path}')
else:
    raise FileNotFoundError(f'링크 파일 없음 — url_수집_colab.ipynb를 먼저 실행하세요\n경로: {links_path}')

# 중간저장 파일 경로 (Drive에 저장해야 Colab 런타임이 끊겨도 이어받기 가능)
checkpoint_name = f"체크포인트_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
checkpoint_path = os.path.join(DRIVE_DIR, checkpoint_name)

# 이전에 중단된 작업이 있으면 이어받기
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    all_results = {int(k): v for k, v in checkpoint['all_results'].items()}
    err_idx = checkpoint['err_idx']
    i = checkpoint['next_i']
    print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
else:
    all_results = dict()
    i = 0
    err_idx = []
    print('새로 시작')

CHECKPOINT_INTERVAL = 100  # 몇 건마다 중간저장할지

for link in naver_news_links[i:]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6)

        # 제목 추출하기
        title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        title = title[0].text

        # 본문 추출하기
        body = driver.find_elements(By.ID, 'newsct_article')
        body = body[0].text.replace('\n', '')

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
        pubdate = pubdate_element[0].get_attribute('data-date-time')

        all_results[i]['link']    = link
        all_results[i]['pubdate'] = pubdate
        all_results[i]['title']   = title
        all_results[i]['body']    = body

        # 진행 상황 확인용 코드
        print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행')
        i += 1

        # 100건마다 중간저장
        if i % CHECKPOINT_INTERVAL == 0:
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False)
            print(f'  >>> 중간저장 완료 ({i}건)')

        # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.5, 1.2))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행 - 오류 발생')
        i += 1

        # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(0.5, 1.2))

# 완료 후 체크포인트 삭제
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print('체크포인트 삭제 완료')

print(len(all_results))
print(err_idx)


In [ ]:
# err_idx가 빈 리스트가 아닐 때만 자동 재시도
# 전체 주석 설정 및 해제하고 싶으면 Ctrl+A로 전체 선택 후 Ctrl+/ 입력

if len(err_idx) != 0:
    print(f'\n오류 {len(err_idx)}건 재시도 시작...')
    re_err_idx = []

    for i in err_idx:
        try:
            link = naver_news_links[i]
            all_results[i] = dict()

            # 실제 네이버 뉴스 웹페이지로 이동
            driver.get(link)

            # 페이지 로딩 대기
            time.sleep(0.6)

            # 제목 추출하기
            title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
            title = title[0].text

            # 본문 추출하기
            body = driver.find_elements(By.ID, 'newsct_article')
            body = body[0].text.replace('\n', '')

            # 날짜 추출하기
            pubdate_element = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
            pubdate = pubdate_element[0].get_attribute('data-date-time')

            all_results[i]['link']    = link
            all_results[i]['pubdate'] = pubdate
            all_results[i]['title']   = title
            all_results[i]['body']    = body

            # 진행 상황 확인용 코드
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 성공')

            # 봇 탐지 방지를 위해 0.5초에서 1.2초 사이에 랜덤한 시간을 기다림
            time.sleep(random.uniform(0.5, 1.2))

        # 오류 발생 시
        except:
            re_err_idx.append(i)
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 실패')
            time.sleep(random.uniform(0.5, 1.2))

    print(f'\n재시도 완료 — 성공: {len(err_idx) - len(re_err_idx)}건 / 재실패: {len(re_err_idx)}건')
    if re_err_idx:
        print(f'재실패 인덱스: {re_err_idx}')
else:
    print('오류 없음 — 재시도 불필요')

In [ ]:
# 수집한 정보들을 dataframe으로 변환
df = pd.DataFrame(all_results).T
df

In [ ]:
# 수집한 기사들 중 중복인 경우 이를 제거
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)
df_no_duplicates

In [ ]:
# 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'])
df_sorted = df_no_duplicates.sort_values(by='pubdate')
df_sorted

In [ ]:
# 수집한 정보들을 csv로 저장 (Google Drive에 저장)
file_name = f"{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.csv"
save_path = os.path.join(DRIVE_DIR, file_name)

df_sorted.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {save_path}')

In [ ]:
# 브라우저 창 닫기
driver.quit()